<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: white; font-family: Georgia, serif; margin: 0; font-size: 2.4em;">
🔍 Session 13B: Data Deep Dive
</h1>
<p style="color: #B8953E; font-size: 1.3em; margin-top: 10px; font-family: Calibri, sans-serif;">
Bivariate Analysis & Feature–Target Relationships
</p>
<p style="color: #B0D0D0; font-size: 0.95em; margin-top: 8px;">
Credit Risk Modelling Programme &nbsp;|&nbsp; MBA Advanced Analytics &nbsp;|&nbsp; 2025–26
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #B8953E; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">📖 Where We Are in the Journey</h3>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
In <strong>Session 13</strong>, we mapped the data universe: shapes, types, missingness, schemas, and basic KPIs.
We know <em>what</em> the data looks like. Now we need to understand <em>how features relate to our target</em> —
who defaults and who doesn’t, and which characteristics separate them.
</p>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
This session is deliberately <strong>visual and intuitive</strong>. We will look at every important feature through
the lens of default risk before Session 14 introduces formal statistical methods. Think of this as building
your “gut feeling” for the data — the kind of instinct that lets a credit analyst glance at an application
and say <em>“this one worries me.”</em>
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #008C8C; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">🗺️ Notebook Roadmap</h3>
<ol style="color: #333; font-size: 1.05em; line-height: 1.8;">
<li><strong>Setup & Data Reload</strong> — Reload the cleaned data from Session 13</li>
<li><strong>Categorical Features vs Default</strong> — Default rates across every category</li>
<li><strong>Numeric Features vs Default</strong> — Box plots, KDE overlays, and statistical comparisons</li>
<li><strong>Outlier Detection & Treatment</strong> — Identifying and handling extreme values</li>
<li><strong>Feature Transformations</strong> — Log transforms, binning, and why they matter</li>
<li><strong>Cross-Feature Interactions</strong> — When two features together tell a richer story</li>
<li><strong>Correlation Landscape</strong> — A visual, intuitive introduction to feature relationships</li>
<li><strong>Feature Scorecard</strong> — Ranking features by their apparent predictive power</li>
</ol>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 1: Setup & Data Reload</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Pick up where Session 13 left off</p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# ── Branding colours ──
NAVY  = '#0B1F3F'
TEAL  = '#008C8C'
GOLD  = '#B8953E'
CORAL = '#E8634A'
LGOLD = '#FDF6E8'
LTEAL = '#E0F2F2'
PURPLE = '#6C5B7B'
PALETTE = [NAVY, TEAL, GOLD, CORAL, PURPLE, '#C06C84']
sns.set_palette(PALETTE)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.titleweight': 'bold',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("\u2705 Libraries loaded.")

In [ ]:
# ── Load and clean data (reproducing Session 13 steps) ──
DATA_DIR = '../data/home-credit-default-risk/'

app = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))

# Clean DAYS_EMPLOYED anomaly (from Session 13)
app['DAYS_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

# Derive KPIs (from Session 13)
app['AGE_YEARS'] = (-app['DAYS_BIRTH'] / 365).round(1)
app['EMPLOYMENT_YEARS'] = (-app['DAYS_EMPLOYED'] / 365).round(1)
app['DEBT_TO_INCOME'] = (app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']).round(2)
app['ANNUITY_BURDEN'] = (app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']).round(4)
app['CREDIT_GOODS_RATIO'] = (app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']).round(3)
app['EXT_SCORE_BLEND'] = app[
    ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
].mean(axis=1).round(4)

print(f"\u2705 Data loaded: {app.shape[0]:,} rows \u00d7 {app.shape[1]} columns")
print(f"   Default rate: {app['TARGET'].mean():.2%}")

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 2: Categorical Features vs Default</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Which categories carry more risk?</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">For every important categorical feature, compute the default rate within each category. Identify which categories are associated with higher or lower default risk. Learn to read a “default rate bar chart” — the single most common visual in credit analytics.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 What is a “Default Rate Bar Chart”?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
Instead of comparing raw counts (which are misleading because of class imbalance), we compute:
<br><br>
<code style="background: white; padding: 4px 8px; border-radius: 4px;">Default Rate = Number of defaults in category / Total applicants in category</code>
<br><br>
A category with a 15% default rate is roughly twice as risky as the overall 8% baseline.
The overall default rate is our <strong>benchmark line</strong> — categories above it are riskier, below it are safer.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 1: Default Rate by Contract Type</h3>
</div>

In [ ]:
# YOUR CODE: Compute and plot the default rate for NAME_CONTRACT_TYPE
# Steps:
#   1. Group by the feature and compute mean of TARGET (this IS the default rate)
#   2. Also compute count per group
#   3. Create a horizontal bar chart
#   4. Add a vertical benchmark line at the overall default rate
#   5. Annotate bars with the rate and count

# Starter:
# grouped = app.groupby('NAME_CONTRACT_TYPE')['TARGET'].agg(['mean', 'count'])


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">The default rate IS the mean of TARGET within each group, since TARGET is 0/1. <code>app.groupby('NAME_CONTRACT_TYPE')['TARGET'].mean()</code> gives you default rates directly.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Revolving loans have a significantly higher default rate</strong> than cash loans. This is intuitive: revolving credit (like credit lines) tends to be used by borrowers who need ongoing access to funds, which can signal financial stress. A credit manager might apply stricter criteria for revolving loan applicants.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 2: Default Rates Across Key Categorical Features</h3>
</div>

In [ ]:
# YOUR CODE: Plot default rates for these 6 categorical features
# Create a 3x2 grid of subplots
# For ORGANIZATION_TYPE, show only the top 15 categories (it has ~58!)

cat_features = [
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',
]


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Create a reusable function! Define <code>plot_default_rate(feature, ax)</code> that takes a feature name and axis, computes <code>groupby(...).mean()</code>, and plots a horizontal bar chart. Then loop through features.</span>
</div>

<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>Key patterns emerging:</strong><br>• <strong>Income type:</strong> “Maternity leave” and “Unemployed” show the highest default rates. “Pensioner” is the safest.<br>• <strong>Education:</strong> “Lower secondary” defaults most; “Academic degree” defaults least. Education is protective.<br>• <strong>Family status:</strong> “Civil marriage” and “Single” are riskier than “Married” or “Widow”.<br>• <strong>Occupation:</strong> “Low-skill labourers” and “Drivers” are the riskiest occupations.<br>These patterns make intuitive business sense. A stable job, higher education, and marriage are all markers of financial stability.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">As a lending manager, these charts are already actionable. You might, for example, flag applications from “Maternity leave” income types for additional review — not to deny them, but to assess whether the loan terms are appropriate. The goal is never to discriminate, but to ensure the loan sets the borrower up for success.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 3: Gender and Age Profile</h3>
</div>

In [ ]:
# YOUR CODE:
# 1. Compute and plot default rate by CODE_GENDER
# 2. Plot age distributions (density) split by TARGET
# 3. Create age bins (20-25, 25-30, ..., 60-70) and compute default rate per bin
#
# Think about: does age matter? Is the relationship linear?


<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Age is one of the strongest predictors.</strong> Younger applicants (20–30) have default rates of 10–12%, while older applicants (55+) default at only 4–5%. The relationship is roughly monotonic: older → lower risk. This likely reflects accumulated financial stability, paid-off debts, and more conservative borrowing behaviour. Gender shows a gap too — but be cautious about using it directly in models due to regulatory and ethical considerations.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 3: Numeric Features vs Default</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">How do continuous variables differ between repaid and defaulted loans?</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Use box plots and KDE (density) plots to compare the distribution of numeric features between defaulters and non-defaulters. Identify features where the two groups are clearly separated versus features where they overlap almost completely.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 How to Read These Plots</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
<strong>Box plots</strong> show the median (line), interquartile range (box), and outliers (dots).
If the boxes for Repaid and Default are in very different positions, the feature is discriminative.
If they overlap completely, the feature is weak.<br><br>
<strong>KDE (density) plots</strong> are smoothed histograms. Where the Repaid curve and Default curve
separate, the feature is useful. Where they overlap, it is not.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 4: Box Plot Comparison — Key Numeric Features</h3>
</div>

In [ ]:
# YOUR CODE: Create box plots comparing Repaid vs Default for these features
numeric_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DEBT_TO_INCOME',
    'ANNUITY_BURDEN',
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    'AGE_YEARS',
    'EMPLOYMENT_YEARS',
    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS',
]

# Use a 4x3 grid. For each feature:
#   1. Split data by TARGET
#   2. Clip at 1st and 99th percentile to remove extreme outliers
#   3. Create side-by-side box plots


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>ax.boxplot([group0_data, group1_data], labels=['Repaid','Default'], patch_artist=True)</code>. Clip outliers with <code>data[feat].quantile(0.99)</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Feature discrimination strength:</strong><br>• <strong>Strong separation:</strong> EXT_SOURCE_2, EXT_SOURCE_3, AGE_YEARS — the boxes are clearly shifted between groups.<br>• <strong>Moderate separation:</strong> EMPLOYMENT_YEARS, ANNUITY_BURDEN, DEBT_TO_INCOME — some shift, but lots of overlap.<br>• <strong>Weak separation:</strong> AMT_INCOME_TOTAL, CNT_CHILDREN, CNT_FAM_MEMBERS — boxes almost identical.<br>Features with strong box separation will likely be important in our models. Those with weak separation might still contribute in combination with other features.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 5: KDE Density Overlays — Seeing the Full Shape</h3>
</div>

In [ ]:
# YOUR CODE: Create KDE density plots for the top 6 features
# Overlay the Repaid (teal) and Default (coral) distributions
# Where the curves separate, the feature has predictive power

top_features = [
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    'AGE_YEARS',
    'EMPLOYMENT_YEARS',
    'DEBT_TO_INCOME',
    'ANNUITY_BURDEN',
]

# Use .plot.kde() for each group


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Filter by target: <code>app[app['TARGET']==0][feat].plot.kde(ax=ax, label='Repaid')</code>. The KDE is a smoothed histogram — it shows the probability density.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">KDE plots are one of the best tools for communicating feature importance to executives. Where the teal and coral curves pull apart, you can point and say: “This is where the model can distinguish good from bad applicants.” EXT_SOURCE_2 and EXT_SOURCE_3 show the clearest separation — these are the features that should anchor any credit-scoring model.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 4: Outlier Detection & Treatment</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Finding and handling extreme values</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Identify outliers in key numeric features using the IQR method and visual inspection. Understand why outlier treatment is necessary for many models, and when to leave outliers alone.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 6: Outlier Audit</h3>
</div>

In [ ]:
# YOUR CODE: Build an outlier report using the IQR method
# For each feature:
#   Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 - Q1
#   Lower fence = Q1 - 1.5 * IQR
#   Upper fence = Q3 + 1.5 * IQR
#   Outliers = values outside the fences

outlier_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME',
    'ANNUITY_BURDEN',
    'AGE_YEARS',
    'EMPLOYMENT_YEARS',
    'CNT_CHILDREN',
]


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">IQR = Q3 - Q1. Lower fence = Q1 - 1.5*IQR. Upper fence = Q3 + 1.5*IQR. Count outliers: <code>(data < lower).sum() + (data > upper).sum()</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>AMT_INCOME_TOTAL has the most outliers</strong> — some applicants report incomes in the millions. These could be data errors, business clients, or genuinely wealthy individuals. For modelling, we typically <em>cap</em> (winsorise) rather than remove outliers, because extreme income is informative (it signals low default risk). Key rule of thumb: <strong>cap at the 99th percentile</strong> unless you have a domain reason to do otherwise.</span>
</div>

<div style="background: #FDE8E5; border-left: 5px solid #E8634A; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #E8634A;">⚠️ Important</strong><br>
<span style="color: #333;"><strong>When NOT to remove outliers:</strong> In credit risk, extreme values often carry real signal. A person with 10 children or extremely high debt-to-income is genuinely riskier. Removing them removes precisely the cases your model needs to learn from. Prefer <strong>capping (winsorising)</strong> or <strong>log transforms</strong> over deletion.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 5: Feature Transformations</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Making skewed features model-friendly</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Apply log transformations and binning to highly skewed features. Understand <em>why</em> transformations help models learn better, using visual before/after comparisons.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #B8953E;">📐 Why Transform Features?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
Many models (logistic regression, PCA, k-means) assume that features have roughly symmetric distributions.
Highly skewed features (like income, where most people earn modestly but a few earn millions) violate
this assumption and can distort model fitting.<br><br>
<strong>Log transform:</strong> Compresses the right tail. Income of 100K and 1M become 5.0 and 6.0 — much closer.<br>
<strong>Binning:</strong> Converts a continuous feature into ordered categories. Useful when the relationship with
the target is non-linear or step-wise.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 7: Log Transform — Before and After</h3>
</div>

In [ ]:
# YOUR CODE: Apply log transform to skewed features and compare before/after
skewed_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
]

# Use np.log1p(x) which computes log(1 + x) safely (handles zeros)
# Compare skewness before and after: data.skew()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>np.log1p(data)</code> instead of <code>np.log(data)</code> — log1p handles zero values safely. Compute skewness with <code>data.skew()</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">The log transform dramatically reduces skewness. AMT_INCOME_TOTAL goes from a skew of ~20 to ~1. This means the distribution is now much more symmetric and bell-shaped. For models like logistic regression, this transformation can meaningfully improve performance because the model can now see the full range of income variation, not just “everyone is normal except a few extreme outliers.”</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 8: Intelligent Binning — When Categories Tell a Better Story</h3>
</div>

In [ ]:
# YOUR CODE:
# 1. Bin EXT_SOURCE_2 into 10 equal-width bins using pd.cut()
# 2. Compute default rate per bin
# 3. Plot as a bar chart with the benchmark line
# 4. Repeat for AMT_INCOME_TOTAL using pd.qcut() (quantile-based bins)
#
# Question: Is the relationship monotonic (steadily increasing/decreasing)?


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>EXT_SOURCE_2 shows a near-perfect monotonic relationship:</strong> as the score increases, default rate drops from ~18% (lowest decile) to ~3% (highest decile). This is exactly the kind of feature that makes a model’s job easy. Income, by contrast, shows a much flatter pattern — higher income helps, but the effect is weaker and less consistent.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 6: Cross-Feature Interactions</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">When two features together tell a richer story</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Explore whether combining two features reveals patterns invisible in either feature alone. This is the first step toward feature engineering — the art of creating new variables.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 9: Age × Income Interaction</h3>
</div>

In [ ]:
# YOUR CODE: Create a 2D heatmap showing default rates by Age x Income
# Steps:
#   1. Bin AGE_YEARS into 5 groups
#   2. Bin AMT_INCOME_TOTAL into 5 quantile groups
#   3. Use pd.pivot_table() to compute mean TARGET for each combination
#   4. Visualise with sns.heatmap()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>pd.cut()</code> for age bins, <code>pd.qcut()</code> for income quintiles, and <code>app.pivot_table(values='TARGET', index='AGE_GROUP', columns='INCOME_GROUP', aggfunc='mean')</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Young + Low Income = Highest Risk.</strong> The top-left corner of the heatmap (young, low income) shows default rates of 12–14%, while the bottom-right (older, high income) shows 3–5%. This interaction effect is <em>stronger</em> than either feature alone. In Session 14, we’ll formalise this kind of analysis with correlation screening and dimensionality reduction.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 10: External Score × Debt-to-Income</h3>
</div>

In [ ]:
# YOUR CODE: Create a scatter plot of EXT_SOURCE_2 vs DEBT_TO_INCOME
# Colour points by TARGET (repaid=teal, default=coral)
# Sample ~10,000 points for performance
# Cap DEBT_TO_INCOME at 15 for cleaner visuals
#
# Question: Where is the "danger zone"?


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;">The scatter plot reveals a clear <strong>risk landscape</strong>: low external score + high leverage = concentrated defaults (bottom-left cluster of red dots). High external score + moderate leverage = almost no defaults. This 2D view is more powerful than either feature alone and previews the kind of multi-feature thinking that underpins all credit-scoring models.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 7: Correlation Landscape</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">A visual, intuitive introduction to feature relationships</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Build a correlation matrix for key numeric features and learn to read it. Identify which features are correlated with TARGET (useful for prediction) and which features are correlated with <em>each other</em> (potential redundancy).</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 11: Correlation with TARGET</h3>
</div>

In [ ]:
# YOUR CODE: Compute correlation of all numeric features with TARGET
# Sort and display the top 15 most positively and negatively correlated
#
# Steps:
#   1. app.select_dtypes(include=[np.number]).corr()['TARGET']
#   2. Sort values
#   3. Show top 15 and bottom 15 in a horizontal bar chart



<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Get all correlations: <code>app.select_dtypes(include=[np.number]).corr()['TARGET']</code>. Sort: <code>.sort_values()</code>. Bottom 15 = head(15), top 15 = tail(15).</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Negative correlation = protective (lower default).</strong> EXT_SOURCE_3 (≈0.18), EXT_SOURCE_2 (≈0.16), and AGE_YEARS (≈0.08) are the strongest protective factors. <strong>Positive correlation = risky.</strong> DAYS_BIRTH (which is negative, so higher values = younger = riskier) and some FLAG_DOCUMENT columns show positive correlation. Most features have very weak correlation (<0.05 in absolute terms). This is normal — credit default is a complex event that no single feature predicts well.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 12: Feature-to-Feature Correlation Heatmap</h3>
</div>

In [ ]:
# YOUR CODE: Build a correlation heatmap for ~15 key features
# Use sns.heatmap() with a triangular mask
# Look for: high feature-target correlations AND high feature-feature correlations

heatmap_features = [
    'TARGET',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME', 'ANNUITY_BURDEN', 'CREDIT_GOODS_RATIO',
    'EXT_SCORE_BLEND',
    'DAYS_EMPLOYED_ANOMALY',
    'CNT_CHILDREN',
]


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>Two things to notice:</strong><br>• <strong>Feature-target correlations</strong> (first column): EXT_SOURCE variables dominate. Everything else is weaker.<br>• <strong>Feature-feature correlations</strong>: AMT_CREDIT and AMT_GOODS_PRICE are highly correlated (r≈0.97). DEBT_TO_INCOME and CREDIT_GOODS_RATIO are also correlated. These <em>redundant</em> features will be handled in Session 14 using variance filters and PCA.</span>
</div>

<div style="background: linear-gradient(135deg, #E8EDF4 0%, #E0F2F2 100%); border: 2px solid #008C8C; padding: 18px 22px; border-radius: 10px; margin: 20px 0;">
<strong style="color: #0B1F3F;">🔀 Why This Matters Next</strong><br>
<span style="color: #333; line-height: 1.6;">In Session 14, we’ll formalise what we’ve seen here. The correlation heatmap will become a tool for <strong>multicollinearity detection</strong>. The feature-target correlations will feed into <strong>variance filters</strong>. And the patterns we noticed (groups of related features) will motivate <strong>PCA and Factor Analysis</strong> — mathematical techniques for discovering the underlying “themes” hidden in 120+ features.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 8: Feature Scorecard</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Ranking features by their apparent predictive power</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Synthesise everything we have learned into a single scorecard that ranks features by their usefulness for predicting default. This is the deliverable that bridges data exploration (Sessions 13 + 13B) to formal modelling (Sessions 14+).</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 13: Build the Feature Scorecard</h3>
</div>

In [ ]:
# YOUR CODE: Build a feature scorecard
# For each numeric feature, compute:
#   1. Absolute correlation with TARGET
#   2. Cohen's d (effect size) = |mean_group0 - mean_group1| / pooled_std
#   3. Missing %
#   4. A combined "Signal Strength" score
#
# Rank features from strongest to weakest signal

numeric_cols_clean = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME', 'ANNUITY_BURDEN', 'CREDIT_GOODS_RATIO',
    'EXT_SCORE_BLEND',
    'DAYS_EMPLOYED_ANOMALY',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
]


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Cohen’s d measures how far apart two group means are, relative to their variability. <code>d = abs(mean0 - mean1) / sqrt((std0**2 + std1**2) / 2)</code>. A d > 0.2 is “small”, > 0.5 is “medium”, > 0.8 is “large”.</span>
</div>

In [ ]:
# YOUR CODE: Visualise the top 20 features from your scorecard
# Use a horizontal bar chart with colour coding:
#   Strong (>5): teal, Moderate (2-5): gold, Weak (<2): coral


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>The scorecard confirms what we’ve seen throughout this session:</strong><br>• <strong>Tier 1 (Strong):</strong> EXT_SOURCE_2, EXT_SOURCE_3, EXT_SCORE_BLEND, AGE_YEARS — these are the pillars of any model.<br>• <strong>Tier 2 (Moderate):</strong> EMPLOYMENT_YEARS, REGION_RATING, DAYS_ID_PUBLISH, ANNUITY_BURDEN — valuable supporting features.<br>• <strong>Tier 3 (Weak):</strong> AMT_INCOME_TOTAL, CNT_CHILDREN, CNT_FAM_MEMBERS — individually weak, but may contribute in ensembles.<br>This ranking will guide feature selection in Session 14 and model building in Sessions 15+.</span>
</div>

In [ ]:
# ══════════════════════════════════════════════════
#  SESSION 13B \u2014 KEY TAKEAWAYS
# ══════════════════════════════════════════════════

print("\u2550" * 70)
print("  SESSION 13B \u2014 DATA DEEP DIVE SUMMARY")
print("\u2550" * 70)

findings = [
    ("Strongest predictors",     "EXT_SOURCE_2, EXT_SOURCE_3, AGE_YEARS"),
    ("Key risk factors",         "Young age, low ext. scores, high leverage"),
    ("Riskiest categories",      "Maternity leave, low-skill labourers, revolving loans"),
    ("Safest categories",        "Pensioners, academic degree, married"),
    ("Best transformation",      "Log transform for income and credit amounts"),
    ("Strongest interaction",    "Young + Low Income = highest default zone"),
    ("Redundant feature pair",   "AMT_CREDIT \u2194 AMT_GOODS_PRICE (r=0.97)"),
    ("Recommended next step",    "Formal feature selection (Session 14)"),
]

print(f"\n\u250C{'\u2500' * 68}\u2510")
for label, value in findings:
    print(f"\u2502  {label:<25s} {value:>40s} \u2502")
print(f"\u2514{'\u2500' * 68}\u2518")

print("\n\u2705 Session 13B Data Deep Dive Complete!")
print("\u27A1\uFE0F  Next: Session 14 \u2014 Signal Extraction (PCA, Factor Analysis, Variance Filters)")

<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 30px; border-radius: 12px; margin-top: 30px;">
<h2 style="color: white; font-family: Georgia, serif; margin: 0;">
✅ Session 13B Complete
</h2>
<p style="color: #B8953E; font-size: 1.15em; margin-top: 10px;">
You now have deep intuition about which features predict default and why. You can articulate feature
importance using default rate charts, KDE plots, and correlation analysis. You have a ranked feature
scorecard that will guide all subsequent modelling work.
</p>
<p style="color: #B0D0D0; font-size: 1em; margin-top: 8px;">
<strong>Next Session Preview:</strong> In Session 14, we’ll formalise this intuition with
<strong>correlation screening</strong> (removing redundant features), <strong>variance filters</strong>
(removing uninformative features), and <strong>PCA vs Factor Analysis</strong> (discovering the hidden
structure in 120+ features). The scorecard you built today is your baseline — Session 14 will
show you whether the math agrees with your gut.
</p>
</div>